In [1]:
import pandas as pd

In [3]:
df = pd.read_csv(
    "API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_285.csv",
    skiprows=4,
    encoding="utf-8-sig"
)

In [4]:
df.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.028282,3.626041,4.257462,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,6.221375,4.689806,4.102851,5.191629,6.824727,10.883478,7.399186,4.770857,4.270111,NaN
2,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186,NaN,NaN
3,Africa Western and Central,AFW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,1.725486,1.784050,1.983092,2.490378,3.745568,7.949251,5.221168,3.608044,1.657818,NaN
4,Angola,AGO,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,29.844480,19.628938,17.080954,22.271539,25.754295,21.355290,13.644102,28.240495,20.162020,NaN


# Data Cleaning

## Handling Missing Values

In [5]:
df.isnull().sum().sum()

np.int64(6283)

In [6]:
df.dropna(axis=1, how="all", inplace=True)

In [7]:
df.isnull().sum().sum()

np.int64(6018)

In [8]:
print(df.isnull().sum().sort_values(ascending=False))

1960              195
1961              193
1962              191
1963              191
1964              186
                 ... 
2011               26
Country Code        0
Indicator Code      0
Indicator Name      0
Country Name        0
Length: 70, dtype: int64


In [9]:
print(
    df.isnull().sum(axis=1).sort_values(ascending=False).head(20)
)

192    66
234    66
227    66
212    66
38     66
198    66
90     66
191    66
50     66
163    66
154    66
148    66
146    66
69     66
136    66
77     66
83     66
109    66
107    66
27     66
dtype: int64


In [10]:
year_cols = [col for col in df.columns if str(col).isdigit()]

missing_by_year = df[year_cols].isnull().sum()

print(missing_by_year)

1960    195
1961    193
1962    191
1963    191
1964    186
       ... 
2021     40
2022     40
2023     42
2024     44
2025     54
Length: 66, dtype: int64


In [11]:
df[year_cols] = df[year_cols].interpolate(
    axis=1,
    method="linear",
    limit_direction="both"
)

In [12]:
df.isnull().sum().sum()

np.int64(1650)

In [13]:
df["Missing_Count"] = df[year_cols].isnull().sum(axis=1)

print(
    df[
        ["Country Name", "Country Code", "Missing_Count"]
    ].sort_values(
        "Missing_Count",
        ascending=False
    ).head(20)
)

                  Country Name Country Code  Missing_Count
69                     Eritrea          ERI             66
148                     Monaco          MCO             66
92                        Guam          GUM             66
27                     Bermuda          BMU             66
234               Turkmenistan          TKM             66
212         Somalia, Fed. Rep.          SOM             66
90                   Greenland          GRL             66
109             Not classified          INX             66
50                        Cuba          CUB             66
154           Marshall Islands          MHL             66
38             Channel Islands          CHI             66
107                Isle of Man          IMN             66
83                   Gibraltar          GIB             66
163   Northern Mariana Islands          MNP             66
254     British Virgin Islands          VGB             66
192  Korea, Dem. People's Rep.          PRK             

In [14]:
df.to_csv(
    "worldbank_inflation_clean_1650.csv",
    index=False
)

print("Dataset saved successfully!")
print("Shape:", df.shape)

Dataset saved successfully!
Shape: (265, 71)


In [15]:
# Remove the helper column if it already exists
df.drop(columns=["Missing_Count"], errors="ignore", inplace=True)

# Identify year columns
year_cols = [
    col for col in df.columns
    if str(col).isdigit()
]

# Count missing values
df["Missing_Count"] = df[year_cols].isnull().sum(axis=1)

# Remove countries with NO inflation data
df_model = df[df["Missing_Count"] < len(year_cols)].copy()

# Remove helper column
df_model.drop(columns=["Missing_Count"], inplace=True)

# Reset index
df_model.reset_index(drop=True, inplace=True)

print("Original rows:", len(df))
print("Rows after removing countries with no data:", len(df_model))
print("Countries removed:", len(df) - len(df_model))

Original rows: 265
Rows after removing countries with no data: 240
Countries removed: 25


In [16]:
df_model.to_csv(
    "worldbank_inflation_clean_countries_removed.csv",
    index=False
)

print("Dataset saved successfully!")
print("Shape:", df.shape)

Dataset saved successfully!
Shape: (265, 71)


In [17]:
df_model["Missing_Count"] = (
    df_model[year_cols]
    .isnull()
    .sum(axis=1)
)

print(
    df_model[
        ["Country Name", "Country Code", "Missing_Count"]
    ]
    .sort_values("Missing_Count", ascending=False)
    .head(30)
)


                    Country Name Country Code  Missing_Count
0                          Aruba          ABW              0
1    Africa Eastern and Southern          AFE              0
152                North America          NAC              0
153                      Namibia          NAM              0
154                New Caledonia          NCL              0
155                        Niger          NER              0
156                      Nigeria          NGA              0
157                    Nicaragua          NIC              0
158                  Netherlands          NLD              0
159                       Norway          NOR              0
160                        Nepal          NPL              0
161                        Nauru          NRU              0
162                  New Zealand          NZL              0
163                 OECD members          OED              0
164                         Oman          OMN              0
165           Other smal

In [18]:
# Make sure Missing_Count is calculated
df_model["Missing_Count"] = df_model[year_cols].isnull().sum(axis=1)

# Show ONLY countries that have missing values
missing_countries = (
    df_model[df_model["Missing_Count"] > 0]
    [["Country Name", "Country Code", "Missing_Count"]]
    .sort_values("Missing_Count", ascending=False)
)

print(missing_countries)

Empty DataFrame
Columns: [Country Name, Country Code, Missing_Count]
Index: []


In [19]:
print("Total missing values:", df_model[year_cols].isnull().sum().sum())

print(
    "Countries with missing values:",
    (df_model["Missing_Count"] > 0).sum()
)

Total missing values: 0
Countries with missing values: 0


In [20]:
df_model.drop(columns=["Missing_Count"], inplace=True)


In [21]:
df_model.to_csv(
    "worldbank_inflation_model_ready.csv",
    index=False
)

print("Saved successfully!")
print("Shape:", df_model.shape)
print("Missing values:", df_model.isnull().sum().sum())

Saved successfully!
Shape: (240, 70)
Missing values: 0


## 